# Tools Playground

Manuální testování catalog tools: `search_products`, `get_product_details`, `check_allergens`, `user_allergens`.

**Předpoklad**: `data/products.json` a `data/chroma/` existují. Pokud ne, spusť:
```bash
uv run python -m scripts.generate_catalog
```

In [1]:
from kosik_workshop.config import load_env
load_env()

from kosik_workshop.catalog.schema import Allergen
from kosik_workshop.tools import (
    search_products,
    get_product_details,
    check_allergens,
    user_allergens,
    set_default_user_allergens,
    ALL_TOOLS,
)

# LangChain tool spec se dá zobrazit
for t in ALL_TOOLS:
    print(f"{t.name:24s} args={list(t.args.keys())}")

search_products          args=['query', 'category', 'max_price_czk', 'vegan_only', 'k']
get_product_details      args=['product_id']
check_allergens          args=['product_id', 'user_allergens']
user_allergens           args=[]


## 1. `search_products` — sémantické hledání

Tools v LangChainu se volají přes `.invoke({...})`.

In [2]:
results = search_products.invoke({"query": "české pivo", "k": 3})
for r in results:
    print(f"  {r['name']:50s} {r['price_czk']:>6} Kč  [{r['category']}]")

  Budweiser Budvar Lager 0,5 l                         40.0 Kč  [Alkohol]
  Pilsner Urquell Pilsner 0,5 l                        35.0 Kč  [Alkohol]
  Pilsner Urquell Nealko 0,5 l                         45.0 Kč  [Alkohol]


In [3]:
# Filtr podle kategorie
results = search_products.invoke({
    "query": "sladký dezert",
    "category": "Sladkosti a snacky",
    "k": 5,
})
for r in results:
    print(f"  {r['name']:50s} {r['price_czk']:>6} Kč")

  Orion Mléčné řezky 250 g                             75.0 Kč
  Orion Tatranky 200 g                                 45.0 Kč
  Orion Čokoládová tyčinka 50 g                        19.0 Kč
  Orion Rolky 150 g                                    55.0 Kč
  Lindt pralinky 200 g                                299.0 Kč


In [4]:
# Filtr podle ceny + vegan only
results = search_products.invoke({
    "query": "rostlinné mléko",
    "max_price_czk": 80,
    "vegan_only": True,
    "k": 5,
})
for r in results:
    print(f"  {r['name']:50s} {r['price_czk']:>6} Kč  vegan={r['vegan']}")

  Mlýn Pernštejn Celozrnná mouka 1 kg                  30.0 Kč  vegan=True
  Penam Rohlíky klasické 6 ks                          36.0 Kč  vegan=True
  Mlýn Pernštejn Hladká mouka 1 kg                     22.0 Kč  vegan=True
  Pekárna Svoboda Rohlíky s mákem 8 ks                 40.0 Kč  vegan=True
  Odkolek Chléb se semínky 500 g                       65.0 Kč  vegan=True


## 2. `get_product_details`

In [5]:
# Vezmi první produkt z vyhledávání a podívej se na detaily
first = search_products.invoke({"query": "máslo", "k": 1})[0]
details = get_product_details.invoke({"product_id": first["id"]})
import json; print(json.dumps(details, ensure_ascii=False, indent=2))

{
  "id": "madeta-syrove-pomazankove-maslo-150-g",
  "name": "Madeta Sýrové pomazánkové máslo 150 g",
  "category": "Mléčné výrobky a vejce",
  "subcategory": "Sýry",
  "price_czk": 59.0,
  "unit": "g",
  "description": "Pomazánkové máslo s obsahem sýra, perfektní na chleba nebo jako součást chlebíčků.",
  "allergens": [
    "mléko"
  ],
  "vegan": false,
  "country_of_origin": "Česko",
  "in_stock": true,
  "brand": "Madeta"
}


In [6]:
# Neexistující ID
get_product_details.invoke({"product_id": "tohle-neexistuje"})

{'error': 'not_found', 'product_id': 'tohle-neexistuje'}

## 3. `check_allergens` a `user_allergens`

In [7]:
# Simulujeme uživatele s alergií na mléko a lepek
set_default_user_allergens([Allergen.MILK, Allergen.GLUTEN])
print("Aktuální alergeny uživatele:", user_allergens.invoke({}))

Aktuální alergeny uživatele: ['mléko', 'lepek']


In [8]:
# Check proti produktu — použije default user (mléko, lepek)
chleb = search_products.invoke({"query": "chléb", "category": "Pečivo", "k": 1})[0]
check_allergens.invoke({"product_id": chleb["id"]})

{'safe': False,
 'conflicts': ['lepek'],
 'product_allergens': ['lepek'],
 'user_allergens': ['mléko', 'lepek']}

In [9]:
# Explicitní override — uživatel s alergií jen na arašídy
check_allergens.invoke({
    "product_id": chleb["id"],
    "user_allergens": ["arašídy"],
})

{'safe': True,
 'conflicts': [],
 'product_allergens': ['lepek'],
 'user_allergens': ['arašídy']}

## 4. Scénář: najdi bezpečné snídaně pro uživatele s alergií na lepek a mléko

In [10]:
set_default_user_allergens([Allergen.GLUTEN, Allergen.MILK])
my_allergens = user_allergens.invoke({})

candidates = search_products.invoke({"query": "snídaně cereálie ovoce", "k": 10})

print(f"User allergens: {my_allergens}\n")
for c in candidates:
    check = check_allergens.invoke({"product_id": c["id"], "user_allergens": my_allergens})
    flag = "✓" if check["safe"] else f"✗ ({', '.join(check['conflicts'])})"
    print(f"  {flag}  {c['name']:55s} {c['price_czk']:>6} Kč")

User allergens: ['lepek', 'mléko']

  ✗ (mléko)  Olma Jogurt s chia semínky 150 g                          45.0 Kč
  ✓  Rajčata Cherry 250 g                                      35.0 Kč
  ✓  Okurka hadovka 300 g                                      35.0 Kč
  ✗ (lepek, mléko)  Orion Mléčné řezky 250 g                                  75.0 Kč
  ✗ (mléko)  Olma Jogurt s ovocem 150 g                                35.0 Kč
  ✓  Hroznové víno bezsemenné 300 g                            85.0 Kč
  ✗ (lepek, mléko)  Opavia Sušenky BeBe Dobré ráno 250 g                      35.0 Kč
  ✓  Mrkev 1 kg                                                29.0 Kč
  ✓  Rajčata dřeňová 500 g                                     45.0 Kč
  ✗ (lepek)  De Cecco Těstoviny fusilli 500 g                          40.0 Kč
